In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(
    "index_scores_v3_2026.csv",
    dtype={"NCESSCH": "string", "FIPS County Code": "string", "Zip Code": "string"},
)

## School performance: graduation, growth, and proficiency

We keep these outcomes separate so their meaning stays clear. `YEAR = 2025` selects the 2024–25 reporting year. This is a single-year comparison; several years would be needed to show sustained performance.

The DPI tables join on `year` and `agency_code`. We start with all NC schools whose grade-range category includes high school. ODIS is joined later using a **provisional name-and-county match** because the supplied ODIS school IDs are rounded scientific-notation strings.

Sources: [NC DPI School Report Card data and dictionary](https://www.dpi.nc.gov/data-reports/school-report-cards/school-report-card-resources-researchers) and the local `rcd_code_desc.xlsx` code definitions.

In [ ]:
from pathlib import Path
import re

YEAR = 2025
DATA_DIR = Path("src_datasets_2425")
HIGH_BARRIER_PERCENTILE = 80  # National ODIS percentile

# Read strings first to preserve school codes and distinguish missing values.
def read_report(filename):
    return pd.read_excel(DATA_DIR / filename, dtype="string")

location_raw = read_report("rcd_location.xlsx")
all_high_grade_schools = location_raw.loc[
    location_raw["year"].eq(str(YEAR))
    & location_raw["agency_level"].eq("SCH")
    & location_raw["category_code"].isin(["H", "T", "A"]),
    ["year", "agency_code", "name", "county", "city", "zip",
     "grade_span", "category_code", "school_type", "designation_Type"],
].copy()

charter_schools_df = all_high_grade_schools.loc[
    all_high_grade_schools["designation_Type"].eq("C")
].copy()
schools_df = all_high_grade_schools.loc[
    all_high_grade_schools["designation_Type"].ne("C")
].copy()
assert not schools_df.duplicated(["year", "agency_code"]).any()
print(f"{len(schools_df):,} non-charter NC schools have a grade range that includes high school.")
display(schools_df.head())

### Rate values and masking

`pct` is a percentage, and `den` is the number of students in the relevant cohort or testing group. A rate is not an average exam score.

Masking codes: `1` means **>95%**, `2` means **<5%**, `3` means **fewer than 10**, and `4` means **insufficient data**. Blank or `0` indicates no masking. We retain the labels and denominators, but leave the exact numeric percentage missing when it is masked. In particular, a reported >95% is never changed to an exact 95% or 100%.

In [ ]:
def prepare_rate(data, prefix):
    """Return one outcome with its exact rate, display label, size, and mask."""
    result = data[["year", "agency_code"]].copy()
    values = pd.to_numeric(data["pct"], errors="coerce")
    masks = data["masking"].fillna("0")
    labels = values.round(1).astype("string") + "%"
    mask_labels = {
        "1": ">95%", "2": "<5%", "3": "Suppressed (<10)",
        "4": "Insufficient data",
    }
    assert masks.isin(["0", *mask_labels]).all(), "Check an unfamiliar masking code."
    for code, label in mask_labels.items():
        labels = labels.mask(masks.eq(code), label)

    result[f"{prefix}_pct"] = values.where(masks.eq("0"))
    result[f"{prefix}_display"] = labels.fillna("Missing")
    result[f"{prefix}_n"] = pd.to_numeric(data["den"], errors="coerce")
    result[f"{prefix}_masking"] = masks
    assert not result.duplicated(["year", "agency_code"]).any()
    return result

### 1. Four-year graduation rate

Select the standard cohort (`STD`) and all students (`ALL`). The extended cohort (`EXT`) is a different graduation measure and is not combined with the standard cohort. `graduation_n` records the cohort denominator, so small cohorts remain identifiable.

In [ ]:
graduation_rate_df = read_report("rcd_acc_cgr.xlsx")
graduation_selected = graduation_rate_df.loc[
    graduation_rate_df["year"].eq(str(YEAR))
    & graduation_rate_df["cgr_type"].eq("STD")
    & graduation_rate_df["subgroup"].eq("ALL")
].copy()
graduation_df = prepare_rate(graduation_selected, "graduation")
display(graduation_df.head())

### 2. Academic growth

Select overall growth (`subject == "ALL"`) for all students. `growth_status` tells us whether a school met or exceeded expected growth; `growth_index` preserves the numeric index for later analysis. Growth and proficiency answer different questions: student progress and attainment of a standard.

In [ ]:
growth_raw = read_report("rcd_acc_eg.xlsx")
growth_df = growth_raw.loc[
    growth_raw["year"].eq(str(YEAR))
    & growth_raw["subject"].eq("ALL")
    & growth_raw["subgroup"].eq("ALL"),
    ["year", "agency_code", "eg_status", "eg_index", "eg_score"],
].rename(columns={
    "eg_status": "growth_status", "eg_index": "growth_index",
    "eg_score": "growth_score",
}).copy()
for column in ["growth_index", "growth_score"]:
    growth_df[column] = pd.to_numeric(growth_df[column], errors="coerce")
assert not growth_df.duplicated(["year", "agency_code"]).any()
display(growth_df.head())

### 3. English II and NC Math 1 proficiency

The large `.txt` file is tab-separated. Read it in chunks and retain only the requested year, all students, the EOC grade group, and `GLP` (grade-level proficiency). `E2` is English II and `M1` is NC Math 1. The percentages are shown separately; no weighted performance score is introduced.

These are the tested groups at each school, not the graduation cohort. Different course-taking patterns can affect comparisons, especially for math.

In [ ]:
proficiency_parts = []
school_codes = set(schools_df["agency_code"])
for chunk in pd.read_csv(
    DATA_DIR / "rcd_acc_pc.txt",
    sep="\t", dtype="string", chunksize=250_000,
):
    keep = (
        chunk["year"].eq(str(YEAR))
        & chunk["standard"].eq("GLP")
        & chunk["grade"].eq("EOC")
        & chunk["subgroup"].eq("ALL")
        & chunk["subject"].isin(["E2", "M1"])
        & chunk["agency_code"].isin(school_codes)
    )
    proficiency_parts.append(chunk.loc[keep].copy())

proficiency_selected = pd.concat(proficiency_parts, ignore_index=True)
english_df = prepare_rate(
    proficiency_selected.loc[proficiency_selected["subject"].eq("E2")], "english_ii"
)
math_df = prepare_rate(
    proficiency_selected.loc[proficiency_selected["subject"].eq("M1")], "math_1"
)
print(f"English II: {len(english_df):,} school records; NC Math 1: {len(math_df):,}.")

### One row per school

Left joins retain schools whose outcomes are missing. `validate="one_to_one"` prevents accidental duplicate rows from unfiltered years, subgroups, or subjects. Masked rates remain in the table with their display labels.

In [ ]:
performance_df = schools_df.copy()
for outcome_df in [graduation_df, growth_df, english_df, math_df]:
    performance_df = performance_df.merge(
        outcome_df, on=["year", "agency_code"], how="left", validate="one_to_one"
    )

performance_columns = [
    "name", "county", "school_type", "graduation_display", "graduation_n",
    "growth_status", "growth_index", "english_ii_display", "english_ii_n",
    "math_1_display", "math_1_n",
]
display(performance_df[performance_columns].head(15))

### Join NC ODIS communities to school outcomes

This match normalizes spelling conventions such as “High”/“High School,” “Co.”/“County,” and “&”/“and,” while requiring the same county. Only unique name/county keys are accepted. Schools without a match are retained for review; fuzzy matches are not automatically accepted.

Each accepted match is **provisional**, not an ID-verified crosswalk. School type, admissions, attendance-boundary availability, and the difference between community demographics and enrolled students still matter when interpreting results.

In [ ]:
def normalize_school_name(value):
    text = str(value).lower().replace("@", " at ").replace("&", " and ").replace("'", "")
    text = re.sub(r"\b(co|cnty)\b", "county", text)
    text = re.sub(r"\bsch\b", "school", text)
    text = re.sub(r"\bhs\b", "high school", text)
    text = re.sub(r"\bhigh\s+school\b", "high", text)
    text = re.sub(r"\b(school|high)\s*$", "", text)
    return re.sub(r"[^a-z0-9]", "", text)


def normalize_county(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower().replace("county", ""))


nc_odis_df = df.loc[df["State"].eq("NC")].copy()
nc_odis_df["match_key"] = (
    nc_odis_df["Name"].map(normalize_school_name) + "|"
    + nc_odis_df["County"].map(normalize_county)
)
charter_keys = (
    charter_schools_df["name"].map(normalize_school_name) + "|"
    + charter_schools_df["county"].map(normalize_county)
)
excluded_charter_rows = nc_odis_df["match_key"].isin(charter_keys).sum()
nc_odis_df = nc_odis_df.loc[~nc_odis_df["match_key"].isin(charter_keys)].copy()
print(f"Excluded {excluded_charter_rows} NC ODIS rows identified as charter schools.")
school_lookup = performance_df.copy()
school_lookup["match_key"] = (
    school_lookup["name"].map(normalize_school_name) + "|"
    + school_lookup["county"].map(normalize_county)
)
# Remove ambiguous keys on either side before matching.
ambiguous_keys = set(school_lookup.loc[
    school_lookup["match_key"].duplicated(keep=False), "match_key"
]) | set(nc_odis_df.loc[
    nc_odis_df["match_key"].duplicated(keep=False), "match_key"
])
school_lookup = school_lookup.loc[~school_lookup["match_key"].isin(ambiguous_keys)]

analysis_df = nc_odis_df.merge(
    school_lookup, on="match_key", how="left", validate="many_to_one", indicator=True
)
analysis_df["match_status"] = analysis_df["_merge"].map({
    "both": "Provisional name/county match", "left_only": "Needs review",
    "right_only": "Not applicable",
}).astype("string")
analysis_df = analysis_df.drop(columns="_merge")
matched = analysis_df["agency_code"].notna()
assert analysis_df.loc[matched, "agency_code"].is_unique
analysis_df["high_barrier"] = (
    pd.to_numeric(analysis_df["Composite Score Percentile Rank"], errors="coerce")
    >= HIGH_BARRIER_PERCENTILE
)
print(f"Matched {matched.sum()} of {len(analysis_df)} NC ODIS rows ({matched.mean():.1%}).")
unmatched_df = analysis_df.loc[~matched, ["Name", "County", "City", "Zip Code"]].copy()
display(unmatched_df.head(10))

### Coverage and the high-barrier school profile

The cutoff below is the **national ODIS percentile**, not a newly computed NC quintile. The selected schools are candidates for comparing outcomes; a high ODIS score alone does not label a school high-performing.

Coverage includes reported bounds (>95% or <5%) but excludes missing or suppressed results. Exact numeric rates remain separate so a later analysis can handle those bounds deliberately.

In [ ]:
def reported_rate(data, prefix):
    return data[f"{prefix}_pct"].notna() | data[f"{prefix}_masking"].isin(["1", "2"])


coverage_rows = []
for label, group in [
    ("All NC ODIS schools", analysis_df),
    (f"National ODIS percentile >= {HIGH_BARRIER_PERCENTILE}", analysis_df.loc[analysis_df["high_barrier"]]),
]:
    has_graduation = reported_rate(group, "graduation")
    has_growth = group["growth_status"].isin(["Met", "Exceeded", "NotMet"])
    has_english = reported_rate(group, "english_ii")
    has_math = reported_rate(group, "math_1")
    coverage_rows.append({
        "group": label, "ODIS schools": len(group),
        "matched": group["agency_code"].notna().sum(),
        "graduation reported": has_graduation.sum(),
        "growth reported": has_growth.sum(),
        "English II reported": has_english.sum(),
        "Math 1 reported": has_math.sum(),
        "all four available": (has_graduation & has_growth & has_english & has_math).sum(),
    })
coverage_df = pd.DataFrame(coverage_rows).set_index("group")
display(coverage_df)

high_barrier_df = analysis_df.loc[
    analysis_df["high_barrier"] & analysis_df["agency_code"].notna()
].copy()
profile_columns = [
    "Name", "County", "Composite Score", "Composite Score Percentile Rank",
    "school_type", "SAB Available", "graduation_display", "graduation_n",
    "growth_status", "english_ii_display", "math_1_display", "match_status",
]

display(high_barrier_df.sort_values("Name")[profile_columns].head(20))

### Add 2024–25 school addresses and coordinates for the interactive map

First join DPI addresses by `year` and `agency_code`. Then match those schools to the local NCES 2024–25 public-school locations file to add latitude and longitude.


In [114]:
addresses_2025 = location_raw.loc[
    location_raw["year"].eq(str(YEAR))
    & location_raw["agency_level"].eq("SCH"),
    ["year", "agency_code", "street_addr", "street_addr2", "city", "state", "zip"],
].rename(columns={
    "street_addr": "address_street",
    "street_addr2": "address_street2",
    "city": "address_city",
    "state": "address_state",
    "zip": "address_zip",
}).copy()
assert not addresses_2025.duplicated(["year", "agency_code"]).any()

map_df = analysis_df.loc[analysis_df["agency_code"].notna()].merge(
    addresses_2025,
    on=["year", "agency_code"],
    how="left",
    validate="one_to_one",
)
assert len(map_df) == analysis_df["agency_code"].notna().sum()
map_df["meets_academic_criteria"] = map_df["agency_code"].isin(ranked["agency_code"])

print(f"Schools with addresses: {map_df['address_street'].notna().sum()} / {len(map_df)}")
display(map_df.loc[
    map_df["Name"].isin(["Cape Fear High", "Midway High"]),
    ["Name", "County", "address_street", "address_city", "address_state", "address_zip"],
])

Schools with addresses: 498 / 498


,Name,County,address_street,address_city,address_state,address_zip
1,Cape Fear High,Cumberland County,4762 Clinton Road,Fayetteville,NC,28312
395,Midway High,Sampson County,15274 Spiveys Corner Highway,Newton Grove,NC,28366


### Match the 2024–25 NCES school locations

The local `Public_School_Locations_2024-25.csv` file provides public-school points for the same school year as the DPI outcomes. Match each DPI school to a **unique** NCES record by normalized school name and county. For names that differ or collide, use the exact street, city, ZIP, and county instead. The coordinates below come from NCES `LAT` and `LON`; `nces_ncessch` and `coordinate_match_method` make the join auditable. The ODIS `NCESSCH` field is rounded in the source file, so it is not used as a join key here.

`coordinate_address_agrees` checks whether NCES and DPI spell the street address identically after removing spaces and punctuation. A disagreement needs review before using a point for a detailed school case study; abbreviations can also cause a disagreement.


In [115]:
NCES_COORDINATES_FILE = Path("Public_School_Locations_2024-25.csv")
nces_points = pd.read_csv(
    NCES_COORDINATES_FILE,
    usecols=["NCESSCH", "NAME", "NMCNTY", "STREET", "CITY", "STATE",
             "ZIP", "LAT", "LON", "SCHOOLYEAR"],
    dtype="string",
)
nces_points = nces_points.loc[
    nces_points["STATE"].eq("NC")
    & nces_points["SCHOOLYEAR"].eq("2024-2025")
].copy()
assert nces_points["NCESSCH"].is_unique


def compact_location(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def address_match_key(street, city, zip_code, county):
    return "|".join([
        compact_location(street), compact_location(city),
        compact_location(zip_code), normalize_county(county),
    ])


nces_points["school_key"] = (
    nces_points["NAME"].map(normalize_school_name) + "|"
    + nces_points["NMCNTY"].map(normalize_county)
)
nces_points["address_key"] = nces_points.apply(
    lambda row: address_match_key(
        row["STREET"], row["CITY"], row["ZIP"], row["NMCNTY"]
    ), axis=1,
)

# Remove any coordinates from a previous run before joining the NCES points.
map_df = map_df.drop(columns=[
    "latitude", "longitude", "geocode_status", "nces_ncessch", "nces_name",
    "nces_street", "nces_city", "nces_zip", "coordinate_match_method",
    "coordinate_source", "coordinate_address_agrees", "school_key", "address_key",
], errors="ignore").copy()
map_df["school_key"] = (
    map_df["name"].map(normalize_school_name) + "|"
    + map_df["county"].map(normalize_county)
)
map_df["address_key"] = map_df.apply(
    lambda row: address_match_key(
        row["address_street"], row["address_city"],
        row["address_zip"], row["county"]
    ), axis=1,
)

# Only unique keys may supply a match. This prevents a shared campus address
# or two similarly named schools from silently selecting the wrong point.
unique_names = nces_points.loc[
    ~nces_points["school_key"].duplicated(keep=False)
].set_index("school_key")["NCESSCH"]
unique_addresses = nces_points.loc[
    ~nces_points["address_key"].duplicated(keep=False)
].set_index("address_key")["NCESSCH"]

map_df["nces_ncessch"] = map_df["school_key"].map(unique_names)
map_df["coordinate_match_method"] = pd.Series(pd.NA, index=map_df.index, dtype="string")
name_match = map_df["nces_ncessch"].notna()
map_df.loc[name_match, "coordinate_match_method"] = "name + county"
address_match = map_df["nces_ncessch"].isna()
map_df.loc[address_match, "nces_ncessch"] = (
    map_df.loc[address_match, "address_key"].map(unique_addresses)
)
map_df.loc[
    address_match & map_df["nces_ncessch"].notna(), "coordinate_match_method"
] = "street + city + ZIP + county"
assert map_df["nces_ncessch"].dropna().is_unique

nces_details = nces_points[[
    "NCESSCH", "NAME", "STREET", "CITY", "ZIP", "LAT", "LON"
]].rename(columns={
    "NCESSCH": "nces_ncessch", "NAME": "nces_name",
    "STREET": "nces_street", "CITY": "nces_city", "ZIP": "nces_zip",
    "LAT": "latitude", "LON": "longitude",
})
map_df = map_df.merge(nces_details, on="nces_ncessch", how="left", validate="one_to_one")
map_df[["latitude", "longitude"]] = map_df[["latitude", "longitude"]].apply(
    pd.to_numeric, errors="coerce"
)
map_df["coordinate_source"] = map_df["nces_ncessch"].map(
    lambda code: "NCES 2024-25" if pd.notna(code) else pd.NA
)
map_df["coordinate_address_agrees"] = map_df.apply(
    lambda row: (
        compact_location(row["address_street"])
        == compact_location(row["nces_street"])
    ) if pd.notna(row["nces_ncessch"]) else pd.NA,
    axis=1,
)
map_df = map_df.drop(columns=["school_key", "address_key"])

assert len(map_df) == analysis_df["agency_code"].notna().sum()
assert map_df["agency_code"].is_unique
print("NCES coordinate matches by method:")
print(map_df["coordinate_match_method"].value_counts(dropna=False).to_string())
print(f"Schools with coordinates: {map_df['latitude'].notna().sum()} / {len(map_df)}")
print(f"Street addresses to review: {(map_df['coordinate_address_agrees'] == False).sum()}")
display(map_df.loc[map_df["coordinate_address_agrees"].eq(False), [
    "Name", "County", "address_street", "nces_street"
]])
display(map_df.loc[
    map_df["Name"].isin(["Cape Fear High", "Midway High", "West Columbus High"]),
    ["Name", "County", "nces_name", "nces_ncessch", "latitude", "longitude",
     "coordinate_match_method", "coordinate_address_agrees"],
])
if map_df["latitude"].isna().any():
    display(map_df.loc[map_df["latitude"].isna(), [
        "Name", "County", "address_street", "address_city", "address_zip"
    ]])

map_df.to_csv("map_data.csv", index=False)


NCES coordinate matches by method:
coordinate_match_method
name + county                   495
street + city + ZIP + county      3
Schools with coordinates: 498 / 498
Street addresses to review: 11


,Name,County,address_street,nces_street
26,Southern Alamance High,Alamance County,631 Southern High School Road,631 Southern High Road
107,John A Holmes High,Chowan County,701 North Oakum Street,800 North Oakum Street
149,City Of Medicine Academy,Durham County,301 Crutchfield St,301 Crutchfield Road
231,Hyde Academy,Hyde County,1430 Main Street,20392 Hwy US 264
284,Olympic High School,Mecklenburg County,4301 Sandy Porter Rd,4301 Sandy Porter Rd Suite A
295,Cato Middle College High School,Mecklenburg County,8120 Grier Rd,8120 Grier Rd Suite 171-B
376,Psrc Early College At Rcc,Robeson County,5170 Fayetteville Road,5160 Fayetteville Road
421,Monroe High,Union County,1501 East Sunset Drive,1 High School Dr
422,Forest Hills High,Union County,701 Old Hwy 74,100 Forest Hills School Rd
458,Wake Young Women'S Leadership Academy,Wake County,303 Ashe Avenue,301 Ashe Avenue


,Name,County,nces_name,nces_ncessch,latitude,longitude,coordinate_match_method,coordinate_address_agrees
1,Cape Fear High,Cumberland County,Cape Fear High,370001100402,35.0345,-78.762,name + county,True
117,West Columbus High,Columbus County,West Columbus High,370096000373,34.3281,-78.924,street + city + ZIP + county,True
395,Midway High,Sampson County,Midway High,370414001676,35.175081,-78.473133,name + county,True


### Example presentation graph: barriers and graduation

Each point represents a provisionally matched NC school. The x-axis is its **national ODIS community barrier percentile** and the y-axis is its **2024–25 four-year graduation rate**. Color shows academic growth. The shaded region is the highest barrier quintile.

A triangle placed at 95% means DPI reported **>95%**, so the exact value is unknown. Schools without a reported graduation result are omitted from this graph. This chart helps find schools to investigate; it does not explain why a school has a particular outcome.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plot_df = analysis_df.loc[analysis_df["agency_code"].notna()].copy()
plot_df["barrier_percentile"] = pd.to_numeric(
    plot_df["Composite Score Percentile Rank"], errors="coerce"
)
plot_df["graduation_plot"] = pd.to_numeric(plot_df["graduation_pct"], errors="coerce")
above_95 = plot_df["graduation_display"].eq(">95%").fillna(False)
plot_df.loc[above_95, "graduation_plot"] = 95
plot_df = plot_df.loc[
    plot_df["barrier_percentile"].notna() & plot_df["graduation_plot"].notna()
].copy()
plot_df["growth_label"] = plot_df["growth_status"].fillna("No growth result")

colors = {
    "Exceeded": "#087E8B",
    "Met": "#315C9B",
    "NotMet": "#D2633A",
    "No growth result": "#8B929A",
}
fig, ax = plt.subplots(figsize=(11, 6))
ax.axvspan(HIGH_BARRIER_PERCENTILE, 100, color="#F2E8D8", alpha=0.7, zorder=0)
ax.axvline(HIGH_BARRIER_PERCENTILE, color="#876C49", linestyle="--", linewidth=1)

for status, color in colors.items():
    group = plot_df.loc[plot_df["growth_label"].eq(status)]
    exact = group["graduation_display"].ne(">95%").fillna(False)
    ax.scatter(
        group.loc[exact, "barrier_percentile"],
        group.loc[exact, "graduation_plot"],
        s=32, color=color, alpha=0.65, linewidth=0, zorder=2,
    )
    ax.scatter(
        group.loc[~exact, "barrier_percentile"],
        group.loc[~exact, "graduation_plot"],
        marker="^", s=68, color=color, alpha=0.9,
        edgecolor="#25313A", linewidth=0.5, zorder=3,
    )

ax.set(xlim=(0, 101), ylim=(0, 103))
ax.set_xlabel("ODIS composite percentile (higher = more community barriers)")
ax.set_ylabel("Four-year graduation rate (%)")
ax.set_title("Graduation and academic growth across NC school communities", loc="left", pad=14)
ax.text(81, 6, "Highest barrier quintile", color="#624D30", fontsize=10)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
legend_items = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=color,
           markeredgecolor="none", markersize=8, label=status)
    for status, color in colors.items()
]
legend_items.append(
    Line2D([0], [0], marker="^", color="none", markerfacecolor="white",
           markeredgecolor="#25313A", markersize=9, label="Reported >95% (shown at 95)")
)
ax.legend(handles=legend_items, title="Academic growth / reporting",
          loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.text(0.01, 0.01,
         "Sources: ODIS v3 (2026 release); NC DPI 2024–25. Provisional name/county matches.",
         fontsize=8, color="#525B63")
fig.tight_layout(rect=(0, 0.04, 0.8, 1))
fig.savefig("barriers_graduation_example.png", dpi=220, bbox_inches="tight")
plt.show()
print(f"Plotted {len(plot_df)} schools; {above_95.sum()} have graduation reported as >95%.")

## Which characteristics differ for the selected schools?

Compare the schools already in `ranked` (currently 20) with **the other matched, non-charter, high-barrier schools** (currently 75). Both groups have national ODIS percentiles of at least 80. These are school-level, equally weighted comparisons, using 2024–25 characteristics.

This preserves the existing selection. “Other” means outside the current `ranked` set; it does **not** mean all those schools perform poorly. In particular, the earlier proficiency filter requires exact percentages and can exclude schools reported as **>95%**. We count these below. Missing outcomes can also prevent selection.

We compare chronic absenteeism, economically disadvantaged student share, beginning teacher share, and school size. These are possible explanations or differences in who attends a school, **not proven causes**. The configuration chart and the grades 9–12 comparison help reveal differences between school models. “Regular School” includes some specialized schools and early colleges.

Sources: local `rcd_chronic_absent.xlsx` (`ALL` students), `rcd_acc_eds.xlsx`, `rcd_eq.xlsx`, `rcd_adm.xlsx`, and `rcd_location.xlsx`. All joins use `year` and `agency_code`. [NC DPI definitions and data guide](https://www.dpi.nc.gov/data-reports/school-report-cards/data-sources-and-information-guide).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import StrMethodFormatter

CHARACTERISTICS_DIR = Path("figures/school_characteristics")
CHARACTERISTICS_DIR.mkdir(parents=True, exist_ok=True)
SELECTED = "Selected schools"
OTHER = "Other high-barrier schools"
GROUPS = [SELECTED, OTHER]
GROUP_COLORS = {SELECTED: "#087E8B", OTHER: "#8795A5"}
FEATURES = [
    ("chronic_absence_pct", "Chronic absenteeism", "% of students"),
    ("economic_disadvantage_pct", "Economically disadvantaged students", "% of students"),
    ("beginning_teacher_pct", "Beginning teachers", "% of teaching staff"),
    ("school_size", "School size", "Average daily membership"),
]

# Use the exact membership of the existing ranked DataFrame, not head(20).
selected_codes = set(ranked["agency_code"])
characteristics_df = high_barrier_df.copy()
characteristics_df["comparison_group"] = np.where(
    characteristics_df["agency_code"].isin(selected_codes), SELECTED, OTHER
)
assert selected_codes.issubset(set(characteristics_df["agency_code"]))
assert characteristics_df["agency_code"].is_unique


def read_characteristic_rows(filename, subgroup=None):
    raw = read_report(filename)
    keep = raw["year"].eq(str(YEAR)) & raw["agency_code"].isin(characteristics_df["agency_code"])
    if subgroup is not None:
        keep &= raw["subgroup"].eq(subgroup)
    rows = raw.loc[keep].copy()
    assert not rows.duplicated(["year", "agency_code"]).any(), filename
    return rows


def prepare_characteristic(rows, source_column, name, scale=1, masking_column=None):
    """Preserve exact values, reported intervals, and missing values separately."""
    out = rows[["year", "agency_code"]].copy()
    values = pd.to_numeric(rows[source_column], errors="coerce") * scale
    masks = (rows[masking_column].fillna("0") if masking_column
             else pd.Series("0", index=rows.index, dtype="string"))
    assert masks.isin(["0", "1", "2", "3", "4"]).all()
    is_percent = name.endswith("_pct")
    if is_percent:
        assert values.dropna().between(0, 100).all(), name
    else:
        assert values.dropna().ge(0).all(), name
    out[name] = values.where(masks.eq("0"))
    out[name + "_lower"] = out[name].copy()
    out[name + "_upper"] = out[name].copy()
    # Conservative interval endpoints; these are not imputed exact rates.
    out.loc[masks.eq("1"), [name + "_lower", name + "_upper"]] = [95, 100]
    out.loc[masks.eq("2"), [name + "_lower", name + "_upper"]] = [0, 5]
    out[name + "_masking"] = masks
    decimals = 1 if is_percent else 0
    labels = out[name].map(lambda v: f"{v:,.{decimals}f}" if pd.notna(v) else "Missing")
    if is_percent:
        labels = labels.where(out[name].isna(), labels + "%")
    out[name + "_display"] = labels.replace({"nan": "Missing"})
    for mask, label in {"1": ">95%", "2": "<5%", "3": "Suppressed", "4": "Insufficient data"}.items():
        out.loc[masks.eq(mask), name + "_display"] = label
    return out


characteristic_sources = [
    # Absence and beginning-teacher shares are fractions; EDS is already 0–100.
    prepare_characteristic(read_characteristic_rows("rcd_chronic_absent.xlsx", "ALL"),
                           "pct", "chronic_absence_pct", 100, "masking"),
    prepare_characteristic(read_characteristic_rows("rcd_acc_eds.xlsx"),
                           "pct_eds", "economic_disadvantage_pct", 1, "pct_eds_masking"),
    prepare_characteristic(read_characteristic_rows("rcd_eq.xlsx"),
                           "pct_beg_teachers", "beginning_teacher_pct", 100),
    prepare_characteristic(read_characteristic_rows("rcd_adm.xlsx"),
                           "avg_student_num", "school_size"),
]
for source in characteristic_sources:
    characteristics_df = characteristics_df.merge(
        source, on=["year", "agency_code"], how="left", validate="one_to_one"
    )
assert len(characteristics_df) == len(high_barrier_df)
print(characteristics_df["comparison_group"].value_counts().reindex(GROUPS).to_string())

# Check an inherited selection limitation without changing ranked.
def passes_using_reported_bounds(data, prefix, threshold):
    return (data[prefix + "_pct"].ge(threshold).fillna(False)
            | (data[prefix + "_masking"].eq("1").fillna(False) & (threshold <= 95)))

meets_with_bounds = (
    characteristics_df["growth_status"].isin(["Met", "Exceeded"])
    & passes_using_reported_bounds(characteristics_df, "graduation", graduation_median)
    & passes_using_reported_bounds(characteristics_df, "english_ii", english_median)
    & passes_using_reported_bounds(characteristics_df, "math_1", math_median)
)
additional_bound_qualifiers = characteristics_df.loc[
    meets_with_bounds & characteristics_df["comparison_group"].eq(OTHER),
    ["Name", "english_ii_display", "math_1_display"],
]
print(f"Selection note: {len(additional_bound_qualifiers)} additional schools meet the benchmarks "
      "when reported >95% proficiency bounds are included. They remain in 'Other' here.")
if len(additional_bound_qualifiers):
    display(additional_bound_qualifiers)

### Coverage and median comparisons

The table reports exact, bounded, and missing values separately. For a masked **<5%** rate, its possible value lies between 0 and 5. A **>95%** rate lies between 95 and 100. Median bounds use these endpoints so masked schools stay in the comparison without assigning them a made-up exact percentage. If both median endpoints agree, the median is determined even though some individual rates are masked. Suppressed and missing values remain excluded and counted.

For percentage measures, the median difference uses **percentage points**. School-size differences use average daily membership, not the graduation cohort. More or less of a characteristic does not by itself mean better or worse.

In [ ]:
def median_bounds(data, column):
    return (data[column + "_lower"].median(), data[column + "_upper"].median())


def format_interval(low, high, is_percent=True):
    if pd.isna(low) or pd.isna(high):
        return "Missing"
    suffix = "%" if is_percent else ""
    if np.isclose(float(low), float(high)):
        return f"{low:,.1f}{suffix}"
    return f"{low:,.1f}–{high:,.1f}{suffix}"


coverage_rows = []
difference_rows = []
for column, label, unit in FEATURES:
    medians = {}
    for group in GROUPS:
        rows = characteristics_df.loc[characteristics_df["comparison_group"].eq(group)]
        exact_n = rows[column].notna().sum()
        reported_n = rows[column + "_lower"].notna().sum()
        low, high = median_bounds(rows, column)
        medians[group] = (low, high)
        coverage_rows.append({
            "Characteristic": label, "Group": group, "Schools": len(rows),
            "Exact": exact_n, "Bounded": reported_n - exact_n,
            "Missing/suppressed": len(rows) - reported_n,
            "Median (or possible range)": format_interval(low, high, column.endswith("_pct")),
        })
    selected_low, selected_high = medians[SELECTED]
    other_low, other_high = medians[OTHER]
    difference_rows.append({
        "Characteristic": label,
        "Selected median": format_interval(selected_low, selected_high, column.endswith("_pct")),
        "Other median": format_interval(other_low, other_high, column.endswith("_pct")),
        "Selected minus other": format_interval(selected_low - other_high, selected_high - other_low, False),
        "Difference unit": "percentage points" if column.endswith("_pct") else "students (ADM)",
    })
characteristic_coverage_df = pd.DataFrame(coverage_rows)
characteristic_comparison_df = pd.DataFrame(difference_rows)
display(characteristic_comparison_df)
display(characteristic_coverage_df)

### Graph 1: How do the groups differ?

Each dot represents one school. Triangles show reported bounds at their boundary, **not exact values**. Black lines show the median or its possible range. Labels report the number of schools with usable data. The same plotting function also produces the more restricted comparison below.

In [ ]:
def save_characteristic_figure(fig, stem):
    for extension in ["png", "pdf"]:
        fig.savefig(CHARACTERISTICS_DIR / f"{stem}.{extension}", dpi=200, bbox_inches="tight")


def plot_characteristic_comparison(data, title, stem, note):
    fig, axes = plt.subplots(2, 2, figsize=(13.8, 10.2))
    rng = np.random.default_rng(2025)  # Reproducible sideways jitter only.
    for ax, (column, label, unit) in zip(axes.flat, FEATURES):
        tick_labels = []
        for x, group in enumerate(GROUPS):
            rows = data.loc[data["comparison_group"].eq(group)]
            xs = x + rng.uniform(-0.16, 0.16, len(rows))
            exact = rows[column].notna().to_numpy()
            ax.scatter(xs[exact], rows.loc[exact, column].astype(float),
                       color=GROUP_COLORS[group], s=42, alpha=0.72, edgecolors="none", zorder=3)
            for mask, marker, boundary in [("2", "v", 5), ("1", "^", 95)]:
                bounded = rows[column + "_masking"].eq(mask).fillna(False).to_numpy(dtype=bool)
                ax.scatter(xs[bounded], np.full(bounded.sum(), boundary), marker=marker,
                           color=GROUP_COLORS[group], s=65, edgecolors="#25313A", linewidths=0.5, zorder=4)
            low, high = median_bounds(rows, column)
            if pd.notna(low):
                ax.hlines([low, high], x - .23, x + .23, color="#162D3D", linewidth=2.4, zorder=5)
                ax.vlines(x, low, high, color="#162D3D", linewidth=3, zorder=5)
            available = rows[column + "_lower"].notna().sum()
            tick_labels.append(f"{group}\n{available}/{len(rows)} reported\nMedian: "
                               + format_interval(low, high, column.endswith("_pct")))
        ax.set_title(label, loc="left", fontsize=14, fontweight="bold", pad=12)
        ax.set_ylabel(unit, fontsize=11)
        ax.set_xticks([0, 1], tick_labels, fontsize=10)
        ax.set_xlim(-0.45, 1.45)
        upper = data[column + "_upper"].max()
        upper = float(upper) if pd.notna(upper) else 1
        ax.set_ylim(-max(upper * .035, .5), max(upper * 1.10, 5))
        ax.grid(axis="y", color="#DFE5E9", linewidth=0.7)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        if column == "school_size":
            ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    fig.suptitle(title, fontsize=20, fontweight="bold", x=.08, ha="left", y=.98)
    legend = [
        Line2D([0], [0], marker="o", linestyle="none", color="#647787", label="Exact reported value"),
        Line2D([0], [0], marker="v", linestyle="none", color="#647787", label="<5% (shown at 5)"),
        Line2D([0], [0], marker="^", linestyle="none", color="#647787", label=">95% (shown at 95)"),
        Line2D([0], [0], color="#162D3D", linewidth=2.5, label="Median or median bounds"),
    ]
    fig.legend(handles=legend, loc="lower left", bbox_to_anchor=(.075, .048), ncol=4, frameon=False, fontsize=10)
    fig.text(.08, .027, note, fontsize=9, color="#435665")
    fig.text(.08, .008, "NC DPI 2024–25; ODIS 2026 release. Provisional matches. Descriptive associations only.",
             fontsize=9, color="#435665")
    fig.tight_layout(rect=(.02, .10, 1, .945), h_pad=2.7, w_pad=3)
    save_characteristic_figure(fig, stem)
    plt.show()


plot_characteristic_comparison(
    characteristics_df,
    f"Characteristics of {len(ranked)} selected schools and their high-barrier peers",
    "01_group_characteristics",
    "All groups are non-charter schools at national ODIS percentile ≥80. 'Other' means outside the current ranked set.",
)

### Graph 2: Individual profiles for the selected schools

Each row is one selected school, in the existing ranking order. The gray vertical line or band is the median (or possible median range) of the other high-barrier schools. Values beside points are the school's actual reported value or bound. This helps identify which characteristics are shared and which differ between successful cases.

In [ ]:
profile_df = (characteristics_df.set_index("agency_code")
              .loc[ranked["agency_code"]].reset_index())
peer_df = characteristics_df.loc[characteristics_df["comparison_group"].eq(OTHER)]
fig, axes = plt.subplots(1, 4, figsize=(19, max(8, .4 * len(profile_df) + 2.5)), sharey=True)
y = np.arange(len(profile_df))
for ax, (column, label, unit) in zip(axes, FEATURES):
    low, high = median_bounds(peer_df, column)
    if pd.notna(low):
        ax.axvspan(low, high, color="#8996A2", alpha=.2)
        ax.axvline(low, color="#647787", linestyle="--", linewidth=1.3)
        if not np.isclose(float(low), float(high)):
            ax.axvline(high, color="#647787", linestyle="--", linewidth=1.3)
    max_value = characteristics_df[column + "_upper"].max()
    max_value = float(max_value) if pd.notna(max_value) else 1
    limit = 112 if column.endswith("_pct") else max_value * 1.18
    for row_index, row in profile_df.iterrows():
        value, marker = row[column], "o"
        mask = row[column + "_masking"]
        if pd.notna(mask) and mask == "2":
            value, marker = 5, "v"
        elif pd.notna(mask) and mask == "1":
            value, marker = 95, "^"
        if pd.notna(value):
            ax.scatter(float(value), row_index, marker=marker, s=55, color=GROUP_COLORS[SELECTED], zorder=3)
            ax.annotate(row[column + "_display"], (float(value), row_index), xytext=(7, 0),
                        textcoords="offset points", va="center", fontsize=9, color="#243D4C")
        else:
            ax.text(limit * .97, row_index, "Missing", ha="right", va="center", fontsize=9, color="#647787")
    title = label.replace("Economically disadvantaged students", "Economically\ndisadvantaged students")
    ax.set_title(title + "\nPeer median: " + format_interval(low, high, column.endswith("_pct")),
                 loc="left", fontsize=12, pad=16)
    ax.set_xlim(-limit * .02, limit)
    ax.set_xlabel(unit, fontsize=10)
    ax.set_yticks(y)
    ax.set_ylim(len(profile_df) - .5, -.5)
    ax.grid(axis="y", color="#E6EAED", linewidth=.7)
    ax.spines[["top", "right"]].set_visible(False)
    if column.endswith("_pct"):
        ax.set_xticks([0, 25, 50, 75, 100])
    else:
        ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axes[0].set_yticklabels(profile_df["Name"], fontsize=10)
fig.text(.03, .045, "Dashed line / gray band: other high-barrier schools' median / median bounds. Triangles: reported bounds, not exact rates.", fontsize=10)
fig.text(.03, .023, "School size is average daily membership. NC DPI 2024–25; ODIS 2026 release. Provisional matches; associations only.", fontsize=10)
fig.tight_layout(rect=(0, .075, 1, .95), w_pad=1.5)
save_characteristic_figure(fig, "02_selected_school_profiles")
plt.show()

### Graph 3: School configuration can affect the comparison

The groups can differ in school type and grades served. A grade span ending in 13 is a useful observed characteristic, but it is **not a verified early-college classification**. Some early colleges serve grades 9–12, and some specialized schools have names without “Early College.” Use these charts as context for interpreting attendance, staffing, and school-size differences.

In [ ]:
configuration_df = characteristics_df.copy()
configuration_df["type_label"] = configuration_df["school_type"].replace({
    "Alternative Educat": "Alternative education"
}).fillna("Unknown")
configuration_df["span_label"] = configuration_df["grade_span"].map({
    "09:12": "Grades 9–12", "09:13": "Grades 9–13"
}).fillna("Other grade spans")
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.7))
configuration_tables = {}
for ax, column, title, order in [
    (axes[0], "type_label", "School type", ["Regular School", "Alternative education"]),
    (axes[1], "span_label", "Grades served", ["Grades 9–12", "Grades 9–13", "Other grade spans"]),
]:
    counts = pd.crosstab(configuration_df["comparison_group"], configuration_df[column]).reindex(GROUPS, fill_value=0)
    categories = [value for value in order if value in counts] + [value for value in counts if value not in order]
    counts = counts[categories]
    configuration_tables[column] = counts
    left = np.zeros(len(GROUPS))
    for category, color in zip(categories, ["#315C79", "#C77542", "#AAB5BD", "#706A91"]):
        shares = counts[category].div(counts.sum(axis=1)).mul(100)
        ax.barh(np.arange(2), shares, left=left, color=color, height=.52, label=category)
        for row_index, (share, count) in enumerate(zip(shares, counts[category])):
            if count:
                ax.text(left[row_index] + share / 2, row_index, f"{share:.0f}%\n({count})",
                        ha="center", va="center", fontsize=10, color="white" if category != "Other grade spans" else "#172B38")
        left += shares.to_numpy()
    ax.set_yticks([0, 1], [f"{g}\n(n={counts.loc[g].sum()})" for g in GROUPS], fontsize=10)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Share of schools (%)")
    ax.set_title(title, loc="left", fontsize=14, fontweight="bold")
    ax.legend(loc="upper left", bbox_to_anchor=(0, -.2), frameon=False, fontsize=10)
    ax.spines[["top", "right", "left"]].set_visible(False)
fig.suptitle("School configuration differs between the groups", x=.04, ha="left", fontsize=19, fontweight="bold")
fig.text(.04, .025, "Regular School can include early colleges and other specialized schools. Grade span alone does not establish admissions or school model.", fontsize=9)
fig.tight_layout(rect=(0, .06, 1, .9), w_pad=2)
save_characteristic_figure(fig, "03_school_configuration")
plt.show()

### Graph 4: Compare a more similar subset

Repeat Graph 1 using only DPI **Regular School** rows with grade span **09:12**. This removes alternative schools and other grade configurations from both groups. It still does not control admissions, school programs, student backgrounds, district practices, or reporting differences. These charts cannot establish which characteristic caused better outcomes.

In [ ]:
similar_configuration_df = characteristics_df.loc[
    characteristics_df["school_type"].eq("Regular School")
    & characteristics_df["grade_span"].eq("09:12")
].copy()
print("Regular schools serving grades 9–12:")
print(similar_configuration_df["comparison_group"].value_counts().reindex(GROUPS, fill_value=0).to_string())
plot_characteristic_comparison(
    similar_configuration_df,
    "Do the differences remain within regular schools serving grades 9–12?",
    "04_regular_grades_9_12",
    "Restricted to DPI Regular School and grade span 09:12. This subset can still contain specialized schools and early colleges.",
)

# Save the exact data behind the graphs so presentation figures remain traceable.
export_columns = ["agency_code", "Name", "County", "comparison_group", "school_type", "grade_span",
                  "Composite Score Percentile Rank"]
for column, _, _ in FEATURES:
    export_columns += [column, column + "_display", column + "_lower", column + "_upper"]
characteristics_df[export_columns].to_csv(CHARACTERISTICS_DIR / "school_characteristics.csv", index=False)
characteristic_comparison_df.to_csv(CHARACTERISTICS_DIR / "group_medians.csv", index=False)
characteristic_coverage_df.to_csv(CHARACTERISTICS_DIR / "data_coverage.csv", index=False)
print(f"Saved four graphs as PNG/PDF plus their data to {CHARACTERISTICS_DIR.resolve()}")